<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp05_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 5: K-Nearest Neighbours (K-NN)
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Write a program for implementation of K-Nearest Neighbours (K-NN) in Python.

## Background
K-NN is a lazy, instance-based learner. To classify a point it finds the **k closest training points** (usually by Euclidean distance `sqrt(sum (x_i - y_i)^2)`) and takes a **majority vote**. Because it is distance-based, **feature scaling is essential**, and `k` controls the bias-variance trade-off (small k -> overfit, large k -> underfit).

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
We use the **Wine** dataset from scikit-learn: 178 wines from 3 cultivars described by 13 chemical measurements.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
data = wine.frame.copy()
data["cultivar"] = data["target"].map(dict(enumerate(wine.target_names)))
data.drop(columns="target").to_csv("wine.csv", index=False)
print("Saved wine.csv")

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("wine.csv")
df.head()

## Step 3: Data Preprocessing
Encode the target, split into train/test (stratified) and **standardise** the features (fit on the training set only).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["cultivar"])
X = df.drop(columns="cultivar")
feature_names = X.columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train_raw)
X_train, X_test = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
print("Classes:", list(le.classes_))
print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df.describe().T)
print("\nMissing values:", df.isnull().sum().sum())
print(df["cultivar"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.countplot(data=df, x="cultivar", ax=axes[0]); axes[0].set_title("Class balance")
sns.boxplot(data=df, x="cultivar", y="proline", ax=axes[1]); axes[1].set_title("Proline by cultivar")
sns.scatterplot(data=df, x="flavanoids", y="color_intensity", hue="cultivar", ax=axes[2]); axes[2].set_title("Flavanoids vs colour intensity")
plt.tight_layout(); plt.show()

from sklearn.decomposition import PCA
pcs = PCA(n_components=2).fit_transform(scaler.transform(X))
plt.scatter(pcs[:, 0], pcs[:, 1], c=y, cmap="viridis", edgecolor="k")
plt.title("PCA projection of the standardised data"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

## Step 5: Model Training
First our own K-NN **from scratch**, then scikit-learn's `KNeighborsClassifier`. We pick `k` using 5-fold cross-validation.

In [ ]:
from collections import Counter

class KNNScratch:
    def __init__(self, k=5):
        self.k = k
    def fit(self, X, y):
        self.X, self.y = np.asarray(X), np.asarray(y)
        return self
    def predict(self, X):
        preds = []
        for x in np.asarray(X):
            dists = np.sqrt(((self.X - x) ** 2).sum(axis=1))      # Euclidean distance to every training point
            nearest = np.argsort(dists)[:self.k]                  # indices of k closest
            preds.append(Counter(self.y[nearest]).most_common(1)[0][0])   # majority vote
        return np.array(preds)

knn_scratch = KNNScratch(k=5).fit(X_train, y_train)
print("From-scratch K-NN test accuracy (k=5):", (knn_scratch.predict(X_test) == y_test).mean())

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

ks = range(1, 26)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train, y_train, cv=5).mean() for k in ks]
best_k = list(ks)[int(np.argmax(cv_scores))]

plt.plot(ks, cv_scores, marker="o")
plt.axvline(best_k, color="r", linestyle="--", label=f"best k = {best_k}")
plt.xlabel("k"); plt.ylabel("5-fold CV accuracy"); plt.title("Choosing k"); plt.legend(); plt.show()

knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_train, y_train)
print("Best k:", best_k)

## Step 6: Model Evaluation
Accuracy, classification report and confusion matrix on the held-out test set. We also show **why scaling matters** by training on unscaled data.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = knn.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"K-NN (k={best_k}) Confusion Matrix"); plt.show()

unscaled = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_raw, y_train)
print(f"Accuracy WITHOUT scaling: {unscaled.score(X_test_raw, y_test):.3f}   |   WITH scaling: {knn.score(X_test, y_test):.3f}")
print("Scratch and sklearn predictions agree:", (KNNScratch(best_k).fit(X_train, y_train).predict(X_test) == y_pred).all())

## Step 7: Prediction on New Data
Classify a new wine sample: scale it with the same scaler, then predict.

In [ ]:
new_wine = X_test_raw.iloc[[0]].copy()      # replace these values with your own measurements
new_wine.iloc[0, 0] = 13.5
print(new_wine.T)
pred = knn.predict(scaler.transform(new_wine))[0]
proba = knn.predict_proba(scaler.transform(new_wine))[0]
print("\nPredicted cultivar:", le.classes_[pred])
print("Class probabilities:", dict(zip(le.classes_, proba.round(2))))

## Conclusion
K-NN with scaled features and a cross-validated k reaches about 95-100% accuracy on the wine data, and our scratch implementation matches scikit-learn. Scaling clearly improves accuracy (proline has a huge numeric range and dominates the distances otherwise). K-NN is simple and effective but slow on large datasets because every prediction scans the full training set.